# 04 — Challenge solution

Debrief only. Same kernel, after the five functions, `tools`, `SYSTEM`, and `run_coding_call` exist.

`line_total` added. It should multiply. 2 * 10 = 20.


In [14]:
shutil.copy(STARTER / "pricing_fixed.py", WORK / "pricing.py")
shutil.copy(STARTER / "totals_buggy.py", WORK / "totals.py")
shutil.copy(STARTER / "test_totals.py", WORK / "test_totals.py")

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "The unit tests fail. Make them pass."},
]
n_lookups = 0

for turn in range(8):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=400,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message)
    for call in message.tool_calls:
        result = run_coding_call(call)
        n_lookups = n_lookups + 1
        preview = result if len(result) < 240 else result[:240] + " ..."
        print(call.function.name, "->", preview.replace("\n", " / "))
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

tests_after = run_tests()
print("n_lookups:", n_lookups)
print(tests_after)

--- turn 1 finish_reason: tool_calls ---
list_dir -> pricing.py, test_pricing.py, test_totals.py, totals.py
run_tests -> ====================================================================== / FAIL: test_two_at_ten (test_totals.TotalsTest.test_two_at_ten) / ---------------------------------------------------------------------- / Traceback (most recent call last): /  ...


--- turn 2 finish_reason: tool_calls ---
read_file -> def line_total(qty, price): /     return qty + price / 
read_file -> import unittest / from totals import line_total /  /  / class TotalsTest(unittest.TestCase): /     def test_two_at_ten(self): /         self.assertEqual(line_total(2, 10), 20) / 
read_file -> def apply_discount(price, percent): /     return price - price * percent / 100 / 
read_file -> import unittest / from pricing import apply_discount /  /  / class PricingTest(unittest.TestCase): /     def test_ten_percent_off_100(self): /         self.assertEqual(apply_discount(100, 10), 90) / 


--- turn 3 finish_reason: tool_calls ---
write_file -> wrote totals.py (51 chars)


--- turn 4 finish_reason: tool_calls ---
run_tests -> ---------------------------------------------------------------------- / Ran 2 tests in 0.000s /  / OK


--- turn 5 finish_reason: stop ---
Fixed `totals.py` so `line_total(qty, price)` multiplies quantity by price instead of adding them.

Tests now pass.
n_lookups: 8
----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK


In [15]:
assert n_lookups >= 1, "the loop should have run at least one tool"
assert "OK" in tests_after, "tests_after should be the passing unittest output"
print("looks good")

looks good
